In [ ]:
!pip install langdetect

In [ ]:
import pandas as pd
import numpy as np
from langdetect import detect_langs, LangDetectException

from google.colab import drive
drive.mount("/content/gdrive")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


# 1 Read-in dataframe

In [ ]:
# read in dataframe, only use the columns: reviewText, summary, overall
df = pd.read_csv("/content/gdrive/MyDrive/NaturalLanguageProcessing/apple.csv", usecols=["reviewText", "summary", "overall"])
print("The original dataframe has {0} rows".format(df.shape[0]))
df.head()

The original dataframe has 66089 rows


,overall,reviewText,summary
0,1.0,The only good thing about these headphones is ...,Dont buy these
1,2.0,These headphones came pretty quick. I can say ...,I suppose...
2,5.0,I was replacing my earbuds and was really hesi...,Awesome!
3,4.0,"They are pretty good. I would buy them again, ...",iPod headphones
4,1.0,These don't even come close to fitting the Iph...,Fits Like a Glove... on OJ Simpson's Hand


In [ ]:
# printing the count of na values and general info about data-types / counts
print(df.isna().sum())
print(df.info())

# looking at min and max value for the rating column
print(df["overall"].min())
print(df["overall"].max())

overall         0
reviewText    124
summary        71
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66089 entries, 0 to 66088
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   overall     66089 non-null  float64
 1   reviewText  65965 non-null  object 
 2   summary     66018 non-null  object 
dtypes: float64(1), object(2)
memory usage: 1.5+ MB
None
1.0
5.0


# 2 Replace na values and star ratings in reviewText

In [ ]:
# look at the na values in summary
print(df[df["summary"].isna()].shape)
df[df["summary"].isna()].head()

(71, 3)


,overall,reviewText,summary
3867,1.0,received a defective Touch Screen. We had to b...,NaN
5044,5.0,I ordered black but,NaN
5592,5.0,Loved this phone too bad I now upgraded to a ip6,NaN
9699,5.0,,NaN
13277,5.0,great phone love it. looks brand new,NaN


In [ ]:
# look at the "star rating" text in the summary column
stars = ["One Star", "Two Stars", "Three Stars", "Four Stars", "Five Stars", "1 Star", "2 Stars", "3 Stars", "4 Stars", "5 Stars"]
df[df["summary"].isin(stars)]

,overall,reviewText,summary
31,5.0,this was a past order but still works fine :D,Five Stars
32,4.0,"very nice, product, fast delivery, i recommend...",Four Stars
33,5.0,"My son loves it, I cant even touch it. LOL",Five Stars
34,1.0,Not good at all!,One Star
35,4.0,good,Four Stars
...,...,...,...
66071,5.0,phone is very good and has more to offer than ...,Five Stars
66073,5.0,The iphone works just fine.,Five Stars
66075,5.0,Love it.,Five Stars
66078,5.0,"works great and delivered on time, just as adv...",Five Stars


In [ ]:
# fill na values in summary with empty string
print("Fill {0} na values with empty string".format(df[df["summary"].isna()].shape[0]))
df.loc[df["summary"].isna(), "summary"] = ""

Fill 71 na values with empty string


In [ ]:
# fill values in summary with only a star rating
print("Fill {0} na values with empty string".format(df[df["summary"].isin(stars)].shape[0]))
df.loc[df["summary"].isin(stars), "summary"] = ""
print("\nDoblecheck:")
indexes = [31, 32, 33, 34, 35, 132, 133, 134, 135, 136]
df.loc[indexes, :]

Fill 24641 na values with empty string

Doblecheck:


,overall,reviewText,summary
31,5.0,this was a past order but still works fine :D,
32,4.0,"very nice, product, fast delivery, i recommend...",
33,5.0,"My son loves it, I cant even touch it. LOL",
34,1.0,Not good at all!,
35,4.0,good,
132,4.0,Great quality pouch fits my samsung galaxy S4 ...,
133,4.0,I love it....Thank You.....!!!!!,
134,2.0,ok,
135,3.0,5 stars,
136,3.0,b,


# 3 Replace na values in reviewText

In [ ]:
# look at the na values in reviewText
print("\nLOOK AT THE NA VALUES IN summary")
print(df[df["reviewText"].isna()].shape)
df[df["reviewText"].isna()].head()


LOOK AT THE NA VALUES IN summary
(124, 3)


,overall,reviewText,summary
3541,5.0,NaN,
3772,5.0,NaN,
4534,5.0,NaN,
4695,5.0,NaN,
4733,5.0,NaN,


In [ ]:
# fill na values in reviewText with empty string
print("Fill {0} na values with empty string".format(df[df["reviewText"].isna()].shape[0]))
df.loc[df["reviewText"].isna(), "reviewText"] = ""
print("\nDoblecheck:")
indexes = [3541, 3772, 4534, 4695, 4733]
df.loc[indexes, :]

Fill 124 na values with empty string

Doblecheck:


,overall,reviewText,summary
3541,5.0,,
3772,5.0,,
4534,5.0,,
4695,5.0,,
4733,5.0,,


In [ ]:
# print the rows that are only empty in the reviewText but not in summary
print("There are {0} rows with empty reviewText and non-empty summary".format(df.loc[(df["reviewText"] == "") & (df["summary"] != "")].shape[0]))
df.loc[(df["reviewText"] == "") & (df["summary"] != "")]

There are 11 rows with empty reviewText and non-empty summary


,overall,reviewText,summary
7682,1.0,,I need help I got the phone it works but my si...
17775,5.0,,Actual apple product. A+
29486,5.0,,It did well!
40057,5.0,,Top
42601,5.0,,Nice and fast shipment
42887,1.0,,Damaged
45839,5.0,,Seller was awesome to my questions
50051,5.0,,Works fine
54463,4.0,,Awaiting feedback from recipient
59304,5.0,,Absolutely good


# 4 Combine reviewText and summary into corpus and drop empty rows

In [ ]:
# combine the reviewText and summary columns
df["corpus"] = df["summary"] + " " + df["reviewText"]
df = df.drop(columns=["reviewText", "summary"])
df.head()

,overall,corpus
0,1.0,Dont buy these The only good thing about these...
1,2.0,I suppose... These headphones came pretty quic...
2,5.0,Awesome! I was replacing my earbuds and was re...
3,4.0,iPod headphones They are pretty good. I would ...
4,1.0,Fits Like a Glove... on OJ Simpson's Hand Thes...


In [ ]:
# drop rows with empty corpus (combined reviewText and summary)
print("There are {0} rows with empty corpus".format(df[df["corpus"] == " "].shape[0]))
df = df[df["corpus"] != " "]

There are 113 rows with empty corpus


In [ ]:
# drop rows that solely contain non-english characters
print("There are {0} rows with only non-english characters".format(df[~df["corpus"].str.contains("[a-zA-Z]")].shape[0]))
df = df[df["corpus"].str.contains("[a-zA-Z]")]

There are 30 rows with only non-english characters


In [ ]:
# drop rows that contain less than 5 words
min_size = 5
print("There are {0} rows with less than {1} words".format(df[df["corpus"].str.split().str.len() < min_size].shape[0], min_size))
print("\nRemoving rows such as:")
print(df[df["corpus"].str.split().str.len() < min_size].head(4))
df = df[df["corpus"].str.split().str.len() >= min_size]
print(f"\nNumber of rows after filtering: {df.shape[0]}")

There are 12644 rows with less than 5 words

Removing rows such as:
     overall                             corpus
34       1.0                   Not good at all!
35       4.0                               good
133      4.0   I love it....Thank You.....!!!!!
134      2.0                                 ok

Number of rows after filtering: 53302


# 5 Check for duplicate rows

In [ ]:
# remove duplicates
print("There are {0} duplicate rows".format(df.duplicated().sum()))
df = df.drop_duplicates()

There are 1603 duplicate rows


# 6 Replace line breaks with white space

In [ ]:
# replace \n with space
print("There are {0} rows where corpus contains '\\n'".format(df[df["corpus"].str.contains("\n")].shape[0]))
df["corpus"] = df["corpus"].str.replace("\n", " ")

There are 4733 rows where corpus contains '\n'


In [ ]:
df.head()

,overall,corpus
0,1.0,Dont buy these The only good thing about these...
1,2.0,I suppose... These headphones came pretty quic...
2,5.0,Awesome! I was replacing my earbuds and was re...
3,4.0,iPod headphones They are pretty good. I would ...
4,1.0,Fits Like a Glove... on OJ Simpson's Hand Thes...


# 7 Filter for english reviews

In [ ]:
# function that detects the language of each review, storing it in a new column "language"
def detect_language(text):
  try: 
    return detect_langs(text)[0].lang
  except LangDetectException:
    return "unknown"
  else:
    print("Else reached")
    return "unknown"

df["language"] = df["corpus"].apply(detect_language)
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 51699 entries, 0 to 66088
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   overall   51699 non-null  float64
 1   corpus    51699 non-null  object 
 2   language  51699 non-null  object 
dtypes: float64(1), object(2)
memory usage: 1.6+ MB


In [ ]:
# drop rows that are not english, drop the language column (not necessary anymore)
print("There are {0} rows that are not english".format(df[df["language"] != "en"].shape[0]))
df = df[df["language"] == "en"]
df = df.drop("language", axis=1)

There are 886 rows that are not english


In [ ]:
df.head()

,overall,corpus
0,1.0,Dont buy these The only good thing about these...
1,2.0,I suppose... These headphones came pretty quic...
2,5.0,Awesome! I was replacing my earbuds and was re...
3,4.0,iPod headphones They are pretty good. I would ...
4,1.0,Fits Like a Glove... on OJ Simpson's Hand Thes...


# 8 Quick overview of labels

In [ ]:
# print the number of rows that are left after preprocessing
print("After the text-preprocessing there are {0} reviews left in the dataset".format(df["corpus"].shape[0]))

After the text-preprocessing there are 50813 reviews left in the dataset


In [ ]:
# overview of the distribution of labels
df["overall"].value_counts()

5.0    24222
1.0    13785
4.0     5805
3.0     3619
2.0     3382
Name: overall, dtype: int64

In [ ]:
# write the dataframe to a csv that is imported by the other files for further analysis
df.to_csv("/content/gdrive/MyDrive/NaturalLanguageProcessing/apple_text_preprocessed.csv", index=False)